### Coursework: Inventory Management  with Deep Reinforcement Learning
Student:Ashirbaeva Raiana 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score


#### Data loading and preprocessing

In [20]:
df = pd.read_csv('final_inventory_realistic.csv')

# Очистка sales: убираем 'kg' и переводим в числа
df['sales'] = df['sales'].astype(str).str.replace('kg', '').astype(float)
df['date'] = pd.to_datetime(df['date'])

# Разворачиваем таблицу: даты станут строками, а продукты — столбцами
# Если данных за какой-то день нет, ставим 0 (fill_value=0)
product_data = df.pivot_table(index='date', columns='product', values='sales', aggfunc='sum').fillna(0)

# Список всех твоих продуктов для вывода в конце
product_names = product_data.columns.tolist()

print(f"Анализируем продукты: {product_names}")
print(product_data.head())




Анализируем продукты: ['apple', 'banana', 'coffee', 'cola', 'hamburger', 'hotdog', 'juice', 'limonade', 'pizza', 'rice', 'sandwich', 'tea', 'vegetables', 'water']
product     apple  banana  coffee   cola  hamburger  hotdog  juice  limonade  \
date                                                                           
2019-01-01   63.0     0.0    43.0    0.0       65.0     0.0    0.0     124.0   
2019-01-02    0.0     0.0    55.0  107.0        0.0    43.0    0.0       0.0   
2019-01-03    0.0     0.0    27.0    0.0        0.0   101.0    0.0      46.0   
2019-01-04    0.0     0.0     0.0   59.0       56.0     0.0    0.0      55.0   
2019-01-05   47.0    52.0     0.0   27.0       73.0    51.0    0.0     104.0   

product     pizza  rice  sandwich    tea  vegetables  water  
date                                                         
2019-01-01   51.0  57.0       0.0  123.0        36.0   40.0  
2019-01-02   82.0   0.0       0.0    0.0         0.0   42.0  
2019-01-03    0.0   0.0     

In [21]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(product_data)

def create_multi_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data[i:(i + seq_length)])
        ys.append(data[i + seq_length])
    return np.array(xs), np.array(ys)

X, y = create_multi_sequences(scaled_data, 7)
X_train = torch.from_numpy(X).float()
y_train = torch.from_numpy(y).float()

# input_size теперь равен количеству продуктов
n_products = len(product_names)


#### RNN

In [22]:
class MultiProductRNN(nn.Module):
    def __init__(self, input_size, hidden_size=128):
        super(MultiProductRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size) # Выход совпадает с кол-вом продуктов

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

model = MultiProductRNN(input_size=n_products)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.MSELoss()


#### learning

In [23]:
epochs = 100
history_loss = [] #
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    output = model(X_train)
    loss = criterion(output, y_train)
    
    loss.backward()
    optimizer.step()
    
    history_loss.append(loss.item())
    
    if (epoch+1) % 20 == 0:
        print(f'Эпоха [{epoch+1}/{epochs}], Ошибка (Loss): {loss.item():.4f}')



Эпоха [20/100], Ошибка (Loss): 0.0327
Эпоха [40/100], Ошибка (Loss): 0.0161
Эпоха [60/100], Ошибка (Loss): 0.0087
Эпоха [80/100], Ошибка (Loss): 0.0025
Эпоха [100/100], Ошибка (Loss): 0.0011


#### Forecasting results and procurement recommendations

In [24]:
model.eval()
last_7_days_scaled = torch.from_numpy(scaled_data[-7:]).float().view(1, 7, n_products)

with torch.no_grad():
    pred_scaled = model(last_7_days_scaled)
    pred_real = scaler.inverse_transform(pred_scaled.numpy())

print(f"{' НА ОСНОВА ПОСЛЕДНИХ 7ДНЕЙПРОГНОЗ ЗАКУПКИ НА СЛЕДУЮЩУЮ НЕДЕЛЮ ':=^50}")
print(f"{'Продукт':<15} | {'Количество':<12}")
print("-" * 30)

# Проходим по результатам
predictions = pred_real[0] # берем первую (и единственную) строку прогноза
for name, value in zip(product_names, predictions):
    # Округляем: max(0, ...) убирает возможные минусы, round(value) делает целое число
    final_val = max(0, round(value)) 
    print(f"{name.capitalize():<15} | {final_val:<5} кг/ед.")

print("-" * 30)
top_idx = np.argmax(predictions)
print(f"Рекомендация: Обратите внимание на запасы товара '{product_names[top_idx].upper()}',")
print(f"так как  ожидается повышенный спрос.")



 НА ОСНОВА ПОСЛЕДНИХ 7ДНЕЙПРОГНОЗ ЗАКУПКИ НА СЛЕДУЮЩУЮ НЕДЕЛЮ 
Продукт         | Количество  
------------------------------
Apple           | 73    кг/ед.
Banana          | 54    кг/ед.
Coffee          | 42    кг/ед.
Cola            | 122   кг/ед.
Hamburger       | 74    кг/ед.
Hotdog          | 57    кг/ед.
Juice           | 17    кг/ед.
Limonade        | 4     кг/ед.
Pizza           | 0     кг/ед.
Rice            | 0     кг/ед.
Sandwich        | 126   кг/ед.
Tea             | 23    кг/ед.
Vegetables      | 54    кг/ед.
Water           | 135   кг/ед.
------------------------------
Рекомендация: Обратите внимание на запасы товара 'WATER',
так как  ожидается повышенный спрос.


In [25]:
unit_costs = {
    'apple': 85,
    'banana': 145,
    'coffee': 450,
    'cola': 85,
    'hamburger': 180,
    'hotdog': 110,
    'juice': 95,
    'limonade': 65,
    'pizza': 450,
    'rice': 120,
    'sandwich': 140,
    'tea': 150,
    'vegetables': 75,
    'water': 35
}
total_budget = 0
print(f"{' ФИНАНСОВЫЙ ПЛАН ЗАКУПКИ  ':=^45}")
print(f"{'Продукт':<15} | {'Кол-во':<7} | {'Затраты':>10}")
print("-" * 45)


# Извлекаем значения из первого измерения (индекс 0)
for name, value in zip(product_names, pred_real[0]): 
    # Теперь value — это одно число, и float(value) сработает без ошибок
    qty = max(0, round(float(value))) 
    
    price_per_unit = unit_costs.get(name.lower(), 100)
    cost = qty * price_per_unit
    total_budget += cost
    
    print(f"{name.capitalize():<15} | {qty:<7} | {cost:>7} сом")

print("-" * 45)
print(f"ИТОГО К ВЫДЕЛЕНИЮ: {total_budget:,.2f} СОМ")


========= ФИНАНСОВЫЙ ПЛАН ЗАКУПКИ  ==========
Продукт         | Кол-во  |    Затраты
---------------------------------------------
Apple           | 73      |    6205 сом
Banana          | 54      |    7830 сом
Coffee          | 42      |   18900 сом
Cola            | 122     |   10370 сом
Hamburger       | 74      |   13320 сом
Hotdog          | 57      |    6270 сом
Juice           | 17      |    1615 сом
Limonade        | 4       |     260 сом
Pizza           | 0       |       0 сом
Rice            | 0       |       0 сом
Sandwich        | 126     |   17640 сом
Tea             | 23      |    3450 сом
Vegetables      | 54      |    4050 сом
Water           | 135     |    4725 сом
---------------------------------------------
ИТОГО К ВЫДЕЛЕНИЮ: 94,635.00 СОМ


In [ ]:
model.eval()
with torch.no_grad():
    train_preds = model(X_train)
    y_true = y_train.numpy()
    y_pred = train_preds.numpy()
accuracy_r2 = r2_score(y_true, y_pred)

print(f"Точность модели (R2 Score): {accuracy_r2 * 100:.2f}%")

Точность модели (R2 Score): 97.96%
